# Retail E-Commerce Analytics Pipeline
This notebook runs the entire Medallion Architecture pipeline (Bronze, Silver, Gold).

## 1. Configuration & Setup

In [0]:
CATALOG = "retail_demo"
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

BASE_PATH = "/Volumes/retail_demo/raw/retail_files/retail_delta_project"
BATCH_PATH = BASE_PATH
INCR_PATH = BASE_PATH
CHECKPOINT_PATH = f"{BASE_PATH}/_checkpoints"
SCHEMA_PATH = f"{BASE_PATH}/_schemas"

BRONZE_ORDERS = f"{CATALOG}.{RAW_SCHEMA}.bronze_orders"
BRONZE_ORDERS_INCR = f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental"
BRONZE_CUSTOMERS = f"{CATALOG}.{RAW_SCHEMA}.bronze_customers"
BRONZE_CUSTOMERS_CDC = f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_cdc"
BRONZE_PRODUCTS = f"{CATALOG}.{RAW_SCHEMA}.bronze_products"
BRONZE_PRODUCTS_CDC = f"{CATALOG}.{RAW_SCHEMA}.bronze_products_cdc"
BRONZE_STORES = f"{CATALOG}.{RAW_SCHEMA}.bronze_stores"

SILVER_ORDERS_CLEAN = f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean"
SILVER_CUSTOMERS_CLEAN = f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean"
SILVER_PRODUCTS_CLEAN = f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean"
SILVER_STORES_CLEAN = f"{CATALOG}.{SILVER_SCHEMA}.dim_store"

DIM_CUSTOMER_SCD2 = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
DIM_PRODUCT_SCD2 = f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"

FACT_ORDERS = f"{CATALOG}.{GOLD_SCHEMA}.fact_orders"
GOLD_DAILY_SALES = f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"
GOLD_CATEGORY_SALES = f"{CATALOG}.{GOLD_SCHEMA}.gold_category_sales"
GOLD_SEGMENT_SALES = f"{CATALOG}.{GOLD_SCHEMA}.gold_segment_sales"
GOLD_REGION_SALES = f"{CATALOG}.{GOLD_SCHEMA}.gold_region_sales"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
print("Catalog and Schemas verified/created successfully.")

Catalog and Schemas verified/created successfully.


## 2. Bronze Layer (Raw Ingestion)

In [0]:
from pyspark.sql.functions import current_timestamp, col, lit

def ingest_batch_file(file_name, target_table):
    file_path = f"{BATCH_PATH}/{file_name}"
    print(f"Reading batch file from: {file_path}")
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "false").load(file_path)
    df_with_metadata = df.withColumn("source_file", col("_metadata.file_path")).withColumn("ingestion_ts", current_timestamp()).withColumn("load_type", lit("batch"))
    df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(target_table)
    print(f"Successfully ingested {file_name} into {target_table}.")

def ingest_incremental_files(file_pattern, target_table, stream_name):
    checkpoint_dir = f"{CHECKPOINT_PATH}/{stream_name}"
    schema_dir = f"{SCHEMA_PATH}/{stream_name}"
    path_with_pattern = f"{INCR_PATH}/{file_pattern}"
    
    print(f"Starting incremental stream for: {path_with_pattern} into {target_table}")
    df = spark.readStream.format("cloudFiles").option("cloudFiles.format", "csv").option("cloudFiles.schemaLocation", schema_dir).option("cloudFiles.inferColumnTypes", "false").option("header", "true").load(path_with_pattern)
    df_with_metadata = df.withColumn("source_file", col("_metadata.file_path")).withColumn("ingestion_ts", current_timestamp()).withColumn("load_type", lit("incremental"))
    
    query = df_with_metadata.writeStream.format("delta").outputMode("append").option("checkpointLocation", checkpoint_dir).option("mergeSchema", "true").trigger(availableNow=True).toTable(target_table)
    query.awaitTermination()
    print(f"Finished incremental run for {target_table}.")

# Run Bronze Ingestion
ingest_batch_file("orders_batch.csv", BRONZE_ORDERS)
ingest_batch_file("customers_batch.csv", BRONZE_CUSTOMERS)
ingest_batch_file("products_batch.csv", BRONZE_PRODUCTS)
ingest_batch_file("stores_batch.csv", BRONZE_STORES)

ingest_incremental_files("orders_incremental_*.csv", BRONZE_ORDERS_INCR, "bronze_orders_incr")
ingest_incremental_files("customers_cdc_*.csv", BRONZE_CUSTOMERS_CDC, "bronze_customers_cdc")
ingest_incremental_files("products_cdc_*.csv", BRONZE_PRODUCTS_CDC, "bronze_products_cdc")

Reading batch file from: /Volumes/retail_demo/raw/retail_files/retail_delta_project/orders_batch.csv
Successfully ingested orders_batch.csv into retail_demo.raw.bronze_orders.
Reading batch file from: /Volumes/retail_demo/raw/retail_files/retail_delta_project/customers_batch.csv
Successfully ingested customers_batch.csv into retail_demo.raw.bronze_customers.
Reading batch file from: /Volumes/retail_demo/raw/retail_files/retail_delta_project/products_batch.csv
Successfully ingested products_batch.csv into retail_demo.raw.bronze_products.
Reading batch file from: /Volumes/retail_demo/raw/retail_files/retail_delta_project/stores_batch.csv
Successfully ingested stores_batch.csv into retail_demo.raw.bronze_stores.
Starting incremental stream for: /Volumes/retail_demo/raw/retail_files/retail_delta_project/orders_incremental_*.csv into retail_demo.raw.bronze_orders_incremental
Finished incremental run for retail_demo.raw.bronze_orders_incremental.
Starting incremental stream for: /Volumes/ret

## 3. Silver Layer Stage 1 (Cleaning)

In [0]:
from pyspark.sql.functions import col, regexp_replace, row_number, when, expr
from pyspark.sql.window import Window

# Clean Orders
df_batch_orders = spark.read.table(BRONZE_ORDERS)
df_incr_orders = spark.read.table(BRONZE_ORDERS_INCR)
df_combined_orders = df_batch_orders.unionByName(df_incr_orders, allowMissingColumns=True)

df_cleaned_orders = df_combined_orders \
    .withColumn("order_ts", expr("try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss')")) \
    .withColumn("unit_price_clean", regexp_replace(col("unit_price"), r"[^0-9\.]", "")) \
    .withColumn("unit_price", expr("try_cast(unit_price_clean AS double)")) \
    .withColumn("quantity", expr("try_cast(quantity AS integer)")) \
    .withColumn("discount_pct", expr("try_cast(discount_pct AS double)")) \
    .withColumn("gross_amount", expr("try_cast(gross_amount AS double)")) \
    .drop("unit_price_clean")

df_valid_orders = df_cleaned_orders.filter(col("order_id").isNotNull() & col("customer_id").isNotNull())
window_spec_orders = Window.partitionBy("order_id").orderBy(col("ingestion_ts").desc())
df_deduped_orders = df_valid_orders.withColumn("rn", row_number().over(window_spec_orders)).filter(col("rn") == 1).drop("rn")
df_deduped_orders.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(SILVER_ORDERS_CLEAN)
print("Orders cleaned.")

# Clean Customers
df_batch_cust = spark.read.table(BRONZE_CUSTOMERS)
df_cdc_cust = spark.read.table(BRONZE_CUSTOMERS_CDC)
df_combined_cust = df_batch_cust.unionByName(df_cdc_cust, allowMissingColumns=True)

df_cleaned_cust = df_combined_cust \
    .withColumn("signup_date", expr("try_cast(signup_date AS date)")) \
    .withColumn("segment", when(col("segment").isNull() | (col("segment") == ""), "Unknown").otherwise(col("segment"))) \
    .withColumn("city", when(col("city").isNull() | (col("city") == ""), "Unknown").otherwise(col("city")))

df_valid_cust = df_cleaned_cust.filter(col("customer_id").isNotNull())
window_spec_cust = Window.partitionBy("customer_id").orderBy(col("ingestion_ts").desc())
df_deduped_cust = df_valid_cust.withColumn("rn", row_number().over(window_spec_cust)).filter(col("rn") == 1).drop("rn")
df_deduped_cust.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(SILVER_CUSTOMERS_CLEAN)
print("Customers cleaned.")

# Clean Products
df_batch_prod = spark.read.table(BRONZE_PRODUCTS)
df_cdc_prod = spark.read.table(BRONZE_PRODUCTS_CDC)
df_combined_prod = df_batch_prod.unionByName(df_cdc_prod, allowMissingColumns=True)

df_cleaned_prod = df_combined_prod \
    .withColumn("created_date", expr("try_cast(created_date AS date)")) \
    .withColumn("unit_price_clean", regexp_replace(col("unit_price"), r"[^0-9\.]", "")) \
    .withColumn("unit_price", expr("try_cast(unit_price_clean AS double)")) \
    .drop("unit_price_clean")

df_valid_prod = df_cleaned_prod.filter(col("product_id").isNotNull())
window_spec_prod = Window.partitionBy("product_id").orderBy(col("ingestion_ts").desc())
df_deduped_prod = df_valid_prod.withColumn("rn", row_number().over(window_spec_prod)).filter(col("rn") == 1).drop("rn")
df_deduped_prod.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(SILVER_PRODUCTS_CLEAN)
print("Products cleaned.")

# Clean Stores
df_batch_stores = spark.read.table(BRONZE_STORES)
df_valid_stores = df_batch_stores.filter(col("store_id").isNotNull())
window_spec_stores = Window.partitionBy("store_id").orderBy(col("ingestion_ts").desc())
df_deduped_stores = df_valid_stores.withColumn("rn", row_number().over(window_spec_stores)).filter(col("rn") == 1).drop("rn")
df_deduped_stores.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(SILVER_STORES_CLEAN)
print("Stores cleaned.")

Orders cleaned.
Customers cleaned.
Products cleaned.
Stores cleaned.


## 4. Silver Layer Stage 2 (SCD Type 2)

In [0]:
from pyspark.sql.functions import sha2, concat_ws, lit, expr, to_date

# Customers SCD2
df_source_cust = spark.read.table(SILVER_CUSTOMERS_CLEAN)
attr_cols_cust = ["customer_name", "city", "segment", "gender", "signup_date", "status"]
df_stg_cust = df_source_cust.withColumn("hash_value", sha2(concat_ws("||", *[col(c).cast("string") for c in attr_cols_cust]), 256))

if not spark.catalog.tableExists(DIM_CUSTOMER_SCD2):
    df_init_cust = df_stg_cust.withColumn("effective_start_date", col("ingestion_ts").cast("date")).withColumn("effective_end_date", to_date(lit("9999-12-31"), "yyyy-MM-dd")).withColumn("is_current", lit(True)).withColumn("customer_sk", expr("uuid()"))
    df_init_cust.write.format("delta").saveAsTable(DIM_CUSTOMER_SCD2)
else:
    df_stg_cust.withColumn("effective_start_date", col("ingestion_ts").cast("date")).createOrReplaceTempView("source_customers")
    spark.sql(f"""
        MERGE INTO {DIM_CUSTOMER_SCD2} t USING source_customers s ON t.customer_id = s.customer_id AND t.is_current = true
        WHEN MATCHED AND t.hash_value <> s.hash_value THEN UPDATE SET t.effective_end_date = date_sub(s.effective_start_date, 1), t.is_current = false
    """)
    spark.sql(f"""
        MERGE INTO {DIM_CUSTOMER_SCD2} t USING source_customers s ON t.customer_id = s.customer_id AND t.is_current = true AND t.hash_value = s.hash_value
        WHEN NOT MATCHED THEN INSERT (customer_sk, customer_id, customer_name, city, segment, gender, signup_date, status, source_file, ingestion_ts, load_type, hash_value, effective_start_date, effective_end_date, is_current)
        VALUES (uuid(), s.customer_id, s.customer_name, s.city, s.segment, s.gender, s.signup_date, s.status, s.source_file, s.ingestion_ts, s.load_type, s.hash_value, s.effective_start_date, to_date('9999-12-31', 'yyyy-MM-dd'), true)
    """)
print("Customer SCD2 complete.")

# Products SCD2
df_source_prod = spark.read.table(SILVER_PRODUCTS_CLEAN)
attr_cols_prod = ["product_name", "category", "brand", "unit_price", "status", "created_date"]
df_stg_prod = df_source_prod.withColumn("hash_value", sha2(concat_ws("||", *[col(c).cast("string") for c in attr_cols_prod]), 256))

if not spark.catalog.tableExists(DIM_PRODUCT_SCD2):
    df_init_prod = df_stg_prod.withColumn("effective_start_date", col("ingestion_ts").cast("date")).withColumn("effective_end_date", to_date(lit("9999-12-31"), "yyyy-MM-dd")).withColumn("is_current", lit(True)).withColumn("product_sk", expr("uuid()"))
    df_init_prod.write.format("delta").saveAsTable(DIM_PRODUCT_SCD2)
else:
    df_stg_prod.withColumn("effective_start_date", col("ingestion_ts").cast("date")).createOrReplaceTempView("source_products")
    spark.sql(f"""
        MERGE INTO {DIM_PRODUCT_SCD2} t USING source_products s ON t.product_id = s.product_id AND t.is_current = true
        WHEN MATCHED AND t.hash_value <> s.hash_value THEN UPDATE SET t.effective_end_date = date_sub(s.effective_start_date, 1), t.is_current = false
    """)
    spark.sql(f"""
        MERGE INTO {DIM_PRODUCT_SCD2} t USING source_products s ON t.product_id = s.product_id AND t.is_current = true AND t.hash_value = s.hash_value
        WHEN NOT MATCHED THEN INSERT (product_sk, product_id, product_name, category, brand, unit_price, status, created_date, source_file, ingestion_ts, load_type, hash_value, effective_start_date, effective_end_date, is_current)
        VALUES (uuid(), s.product_id, s.product_name, s.category, s.brand, s.unit_price, s.status, s.created_date, s.source_file, s.ingestion_ts, s.load_type, s.hash_value, s.effective_start_date, to_date('9999-12-31', 'yyyy-MM-dd'), true)
    """)
print("Product SCD2 complete.")

Customer SCD2 complete.
Product SCD2 complete.


## 5. Gold Layer (Fact & Analytics)

In [0]:
from pyspark.sql.functions import sum as _sum, countDistinct, avg

# Fact Orders
df_orders = spark.read.table(SILVER_ORDERS_CLEAN).withColumn("order_date", to_date(col("order_ts")))
df_customers = spark.read.table(DIM_CUSTOMER_SCD2)
df_products = spark.read.table(DIM_PRODUCT_SCD2)
df_stores = spark.read.table(SILVER_STORES_CLEAN)

fact_with_cust = df_orders.alias("o").join(df_customers.alias("c"), (col("o.customer_id") == col("c.customer_id")) & (col("o.order_date") >= col("c.effective_start_date")) & (col("o.order_date") <= col("c.effective_end_date")), "left_outer").select("o.*", col("c.customer_sk"))
fact_with_prod = fact_with_cust.alias("o").join(df_products.alias("p"), (col("o.product_id") == col("p.product_id")) & (col("o.order_date") >= col("p.effective_start_date")) & (col("o.order_date") <= col("p.effective_end_date")), "left_outer").select("o.*", col("p.product_sk"))
fact_final = fact_with_prod.alias("o").join(df_stores.alias("s"), col("o.store_id") == col("s.store_id"), "left_outer").select("o.order_id", "o.order_ts", "o.order_date", "o.customer_id", "o.customer_sk", "o.product_id", "o.product_sk", "o.store_id", "o.quantity", "o.unit_price", "o.discount_pct", "o.gross_amount", "o.payment_method", "o.order_status")

fact_final.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(FACT_ORDERS)
print("Fact Orders created.")

# Analytics Aggregations
df_fact = spark.read.table(FACT_ORDERS).filter(col("order_status") != "cancelled")

# Daily
df_fact.groupBy("order_date").agg(countDistinct("order_id").alias("total_orders"), _sum("gross_amount").alias("total_revenue"), _sum("quantity").alias("total_units"), avg("gross_amount").alias("average_order_value")).write.format("delta").mode("overwrite").saveAsTable(GOLD_DAILY_SALES)
# Category
df_fact.join(df_products, "product_sk").groupBy("category").agg(countDistinct("order_id").alias("orders"), _sum("gross_amount").alias("revenue"), _sum("quantity").alias("units_sold")).orderBy(col("revenue").desc()).write.format("delta").mode("overwrite").saveAsTable(GOLD_CATEGORY_SALES)
# Segment
df_fact.join(df_customers, "customer_sk").groupBy("segment").agg(countDistinct("customer_sk").alias("unique_customers"), countDistinct("order_id").alias("orders"), _sum("gross_amount").alias("revenue")).write.format("delta").mode("overwrite").saveAsTable(GOLD_SEGMENT_SALES)
# Region
df_fact.join(df_stores, "store_id").groupBy("region").agg(countDistinct("order_id").alias("orders"), _sum("gross_amount").alias("revenue"), _sum("quantity").alias("units")).write.format("delta").mode("overwrite").saveAsTable(GOLD_REGION_SALES)

Fact Orders created.


In [0]:
%sql
SELECT 
    customer_id, 
    customer_name, 
    city, 
    effective_start_date, 
    effective_end_date, 
    is_current 
FROM retail_demo.silver.dim_customer_scd2
WHERE customer_id IN (
    SELECT customer_id 
    FROM retail_demo.silver.dim_customer_scd2 
    GROUP BY customer_id 
    HAVING count(*) > 1
)
ORDER BY customer_id, effective_start_date;

customer_id,customer_name,city,effective_start_date,effective_end_date,is_current
C00001,Customer_1,New Fake City,2026-08-08,9999-12-31,true
C00001,Customer_1,Delhi,2026-08-08,2026-08-07,false


In [0]:
%sql
-- 1. Simulate a customer moving to a new city in the source data
UPDATE retail_demo.silver.silver1_customers_clean 
SET city = 'New Fake City', ingestion_ts = current_timestamp()
WHERE customer_id = 'C001';

num_affected_rows
0


In [0]:
# 1. Grab a real customer ID from the table
valid_cust_id = spark.read.table("retail_demo.silver.silver1_customers_clean").select("customer_id").first()[0]
print(f"Updating real customer ID: {valid_cust_id}")

# 2. Run the update dynamically!
spark.sql(f"""
    UPDATE retail_demo.silver.silver1_customers_clean 
    SET city = 'New Fake City', ingestion_ts = current_timestamp()
    WHERE customer_id = '{valid_cust_id}'
""")

Updating real customer ID: C00001


DataFrame[num_affected_rows: bigint]